In [1]:
import requests
import json
import csv
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
BASE_URL = "https://web.archive.org/cdx/search/cdx"
FIELDS = ["original", "timestamp", "statuscode", "mimetype"]

SESSION = requests.Session()
SESSION.headers.update(
    {"User-Agent": "OnTheRecord-research/1.0 (+contact: marcus.presutti.eu@gmail.com)"}
)


def _cdx_request(params, max_retries=6):
    """GET one CDX request, retrying on 429/503/504/connection errors with backoff."""
    for attempt in range(max_retries):
        try:
            resp = SESSION.get(BASE_URL, params=params, timeout=300)
        except (requests.ConnectionError, requests.Timeout):
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)
            continue
        if resp.status_code in (429, 503, 504):
            if attempt == max_retries - 1:
                resp.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        resp.raise_for_status()
        return resp
    raise RuntimeError("unreachable")


def _num_pages(domain, page_size):
    """Number of CDX pages for a domain prefix at the given pageSize.

    numPages depends on pageSize, so it MUST be requested with the same
    pageSize used to fetch — otherwise the page range overshoots and the
    server returns 400 for out-of-range pages.
    """
    resp = _cdx_request({"url": f"{domain}/*", "pageSize": page_size,
                         "showNumPages": "true"})
    return int(resp.text.strip())


def _fetch_page(domain, page, page_size):
    """Fetch one page of index blocks, return raw CDX rows (header stripped)."""
    params = {
        "url": f"{domain}/*",
        "output": "json",
        "fl": ",".join(FIELDS),
        "page": page,
        "pageSize": page_size,
    }
    resp = _cdx_request(params)
    try:
        data = resp.json()
    except requests.exceptions.JSONDecodeError:
        data = _parse_cdx_text(resp.text)
    return [r for r in data[1:] if len(r) == len(FIELDS)]


def fetch_wayback_urls(domain, page_size=5, workers=5):
    """Fetch all archived URLs for a domain via parallel CDX block pagination.

    `collapse=urlkey` and resume-key streaming both 504 on huge domains because
    the server scans the whole prefix. The page/pageSize API walks precomputed
    index blocks, each bounded, so pages can be fetched concurrently. Keep
    workers modest (~5): too many concurrent hits gets the IP rate-limited
    (connection refused / 429). Deduplicated client-side on the archived URL.
    """
    total_pages = _num_pages(domain, page_size)
    print(f"  {total_pages} pages (pageSize={page_size}), {workers} workers")

    seen = set()
    rows = []
    done = 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_fetch_page, domain, p, page_size): p
                   for p in range(total_pages)}
        for fut in as_completed(futures):
            for r in fut.result():
                key = r[0]
                if key in seen:
                    continue
                seen.add(key)
                rows.append(dict(zip(FIELDS, r)))
            done += 1
            if done % 25 == 0 or done == total_pages:
                print(f"  {done}/{total_pages} pages -> {len(rows)} unique")

    return rows


def _parse_cdx_text(text):
    """Salvage rows from a truncated JSON array by parsing line by line."""
    out = []
    for line in text.splitlines():
        line = line.strip().rstrip(",")
        if not line or line in ("[", "]"):
            continue
        try:
            out.append(json.loads(line))
        except json.JSONDecodeError:
            continue  # drop the one truncated line
    return out

In [3]:
URLS = {
    # "bt_A" : "bt.dk/politik",
    # "bt_B" : "bt.dk/arkiv",
    # "dr"   : "dr.dk/nyheder/politik",
    "tv2"  : "nyheder.tv2.dk",
    "eb"   : "ekstrabladet.dk/nyheder"
}

# Fetch Wayback URLs for each domain (block-paginated) and write to CSV
for key, url in URLS.items():
    print(f"Fetching Wayback URLs for {url}...")
    results = fetch_wayback_urls(url)
    print(f"Found {len(results)} unique URLs")

    with open(f"../data/scraped/articles/{key}_wayback_urls.csv", "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(results)

Fetching Wayback URLs for nyheder.tv2.dk...
  104 pages (pageSize=5), 5 workers
  25/104 pages -> 153307 unique
  50/104 pages -> 382965 unique
  75/104 pages -> 583272 unique
  100/104 pages -> 762157 unique
  104/104 pages -> 807431 unique
Found 807431 unique URLs
Fetching Wayback URLs for ekstrabladet.dk/nyheder...
  89 pages (pageSize=5), 5 workers
  25/89 pages -> 126818 unique
  50/89 pages -> 359522 unique
  75/89 pages -> 525797 unique
  89/89 pages -> 594386 unique
Found 594386 unique URLs
